In [1]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, RAW, LAKE

from commun.spark_utils import creer_bucket, format_tables
creer_bucket("datalake")

Bucket « datalake » créé.


In [2]:
spark = creer_session("S01b")
FMT = format_tables(spark)   # 'delta' (ou 'parquet' si Delta indisponible)
FMT

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040


'delta'

In [ ]:
RAW = "/home/jovyan/work/data/raw"
fichier = f"{RAW}/comptage_multimodal_comptages.csv"

comptages_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .csv(fichier)
    .withColumn("_source", F.lit("comptages_multimodal_comptages.csv"))
    .withColumn("_ingere_le", F.current_timestamp())
)

print(comptages_raw.columns)
comptages_raw.show(3, truncate=False)

#cp du csv vers rustfs tjrs en Bronze

(comptages_raw.write.format(FMT).mode("overwrite")
          .option("overwriteSchema", "true")
          .save(f"{LAKE}/bronze/comptage_multimodal_paris"))
print("bronze/comptage_multimodal_paris :", spark.read.format(FMT).load(f"{LAKE}/bronze/comptage_multimodal_paris").count(), "lignes")



['Identifiant trajectoire', 'Identifiant site', 'Nom du site', 'Date et heure de comptage', 'Mode déplacement', 'Nombre de véhicules', 'Type voie', 'Sens de circulation', 'Trajectoire', 'Coordonnées Géo', '_source', '_ingere_le']
+-----------------------+----------------+---------------------------+-------------------------+--------------------+-------------------+----------------------------+-------------------+-----------+---------------------+----------------------------------+--------------------------+
|Identifiant trajectoire|Identifiant site|Nom du site                |Date et heure de comptage|Mode déplacement    |Nombre de véhicules|Type voie                   |Sens de circulation|Trajectoire|Coordonnées Géo      |_source                           |_ingere_le                |
+-----------------------+----------------+---------------------------+-------------------------+--------------------+-------------------+----------------------------+-------------------+-----------+------

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: aa2f8227-a421-47ca-87a8-c8e0bf14ca26).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- _c0: string (nullable = true)
 |-- _source: string (nullable = true)
 |-- _ingere_le: timestamp (nullable = true)


Data schema:
root
 |-- Identifiant trajectoire: string (nullable = true)
 |-- Identifiant site: string (nullable = true)
 |-- Nom du site: string (nullable = true)
 |-- Date et heure de comptage: string (nullable = true)
 |-- Mode déplacement: string (nullable = true)
 |-- Nombre de véhicules: string (nullable = true)
 |-- Type voie: string (nulla

In [ ]:
print(comptages_raw.columns)
comptages_raw.show(3, truncate=False)

#cp du csv vers rustfs tjrs en Bronze

(comptages_raw.write.format(FMT).mode("overwrite")
          .save(f"{LAKE}/bronze/comptage_multimodal_paris"))
print("bronze/comptage_multimodal_paris :", spark.read.format(FMT).load(f"{LAKE}/bronze/comptage_multimodal_paris").count(), "lignes")



['Identifiant trajectoire', 'Identifiant site', 'Nom du site', 'Date et heure de comptage', 'Mode déplacement', 'Nombre de véhicules', 'Type voie', 'Sens de circulation', 'Trajectoire', 'Coordonnées Géo', '_source', '_ingere_le']
+-----------------------+----------------+---------------------------+-------------------------+--------------------+-------------------+----------------------------+-------------------+-----------+---------------------+----------------------------------+--------------------------+
|Identifiant trajectoire|Identifiant site|Nom du site                |Date et heure de comptage|Mode déplacement    |Nombre de véhicules|Type voie                   |Sens de circulation|Trajectoire|Coordonnées Géo      |_source                           |_ingere_le                |
+-----------------------+----------------+---------------------------+-------------------------+--------------------+-------------------+----------------------------+-------------------+-----------+------